In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import os

! git clone https://github.com/Afeefaa/AWS
# Load dataset with a random 80/20 split
batch_size = 32
img_size = (256, 256)
dataset_dir = '/content/AWS/train'

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)


normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))


AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(256,256,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.6),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # lower LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)

model.save('final_model.keras')

In [ ]:
import numpy as np
import os
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model


class_names = ['Cardboard', 'Glass', 'Metal', 'Paper', 'Plastic', 'trash']

model = load_model('final_model.keras')

test_dir = '/content/AWS/predb'
if not os.path.exists(test_dir):
    print(f"Test directory {test_dir} not found. Using current directory instead.")
    test_dir = '.'


image_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
image_files = [f for f in os.listdir(test_dir) if f.lower().endswith(image_extensions)]
print(f"Found {len(image_files)} images for evaluation.\n")

correct_argmax = 0
correct_threshold = 0
total = 0

for filename in image_files:

    true_index = -1
    for idx, cls in enumerate(class_names):
        if cls.lower() in filename.lower():
            true_index = idx
            break
    if true_index == -1:
        print(f"Skipping {filename} – no known class name in filename")
        continue


    img_path = os.path.join(test_dir, filename)
    img = cv2.imread(img_path)
    if img is None:
        print(f"Could not read {filename}, skipping.")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (256, 256))
    img = img.astype('float32') / 255.0
    img = np.expand_dims(img, axis=0)

    # Predict
    pred = model.predict(img, verbose=0)[0]
    predicted_index = np.argmax(pred)

    if predicted_index == true_index:
        correct_argmax += 1

    total += 1

# Print final accuracy
print(f"\nTotal images evaluated: {total}")
print(f"Accuracy :        {correct_argmax/total:.2%}")
